# Hotel Messages — Full Pipeline
**Данные:** `2026_04_17_messages.xls` · 4 отеля · 65k сообщений

**Pipeline:**
1. Загрузка + очистка HTML
2. Thread detection (GAP_HOURS=4)
3. GPT-4o-mini классификация (PROBLEM / QUESTION / OTHER)
4. TF-IDF + KMeans топики с препроцессингом
5. GPT оценка лучшего k
6. Итоговые таблицы и графики

In [1]:
# !pip install xlrd pymorphy3 openai pyarrow

In [2]:
import pandas as pd
import numpy as np
import re
import json
import pymorphy3
from pathlib import Path
from datetime import timedelta
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────
XLS_PATH        = "data/2026_04_17_messages.xls"
GAP_HOURS       = 4        # новый тред если пауза > 4 часов
BATCH_SIZE      = 20       # тредов на один GPT-запрос
K_RANGE         = range(5, 21)
CATEGORIES      = ["QUESTION", "PROBLEM"]
OPENAI_API_KEY  = None     # или строкой, или os.environ["OPENAI_API_KEY"]

HOTEL_NAMES = {1: "БС74", 2: "МК16", 3: "You&Co", 4: "М73", 5: "О-44"}

## 1. Загрузка и очистка

In [3]:
df_raw = pd.read_excel(XLS_PATH)
print(f"Загружено строк: {len(df_raw)}")
print(f"Колонки: {df_raw.columns.tolist()}")
print(f"Отели: {sorted(df_raw['hotel_id'].unique())}")

# Убираем HTML-теги (<div><b>текст</b></div> → текст)
df_raw['message'] = df_raw['message'].apply(
    lambda x: re.sub(r'<[^>]+>', '', str(x)).strip()
)

# Оставляем только гостевые сообщения (from_admin = NaN)
# Убираем короткие (<5 символов) и тестовые
TEST_WORDS = r'тест|test|бегу|сява|победа|тестовое'
df = df_raw[
    df_raw['from_admin'].isna() &
    (df_raw['message'].str.len() > 5) &
    ~df_raw['message'].str.lower().str.contains(TEST_WORDS, na=False)
].copy()

df['date_add'] = pd.to_datetime(df['date_add'])
df = df.sort_values(['hotel_id', 'booking_id', 'date_add']).reset_index(drop=True)

print(f"\nГостевых сообщений после фильтра: {len(df)}")
print(df.groupby('hotel_id').size().rename('messages').to_frame().rename(index=HOTEL_NAMES))

Загружено строк: 65535
Колонки: ['booking_id', 'hotel_id', 'message', 'from_admin', 'date_add', 'is_admin_read', 'is_user_read']
Отели: [1, 2, 3, 4]

Гостевых сообщений после фильтра: 24021
          messages
hotel_id          
БС74         14714
МК16          5104
You&Co        4107
М73             96


## 2. Thread Detection

Новый тред = пауза между сообщениями одного гостя > `GAP_HOURS` часов.

**Le Wagon:** Data Toolkit → Time Series, groupby + shift

In [4]:
def detect_threads(df, gap_hours=GAP_HOURS):
    """Присваиваем thread_id внутри каждого бронирования."""
    records = []
    for (hotel_id, booking_id), grp in df.groupby(['hotel_id', 'booking_id']):
        grp = grp.sort_values('date_add').reset_index(drop=True)
        thread_id = 1
        prev_time = None
        for _, row in grp.iterrows():
            if prev_time is not None:
                gap = (row['date_add'] - prev_time).total_seconds() / 3600
                if gap > gap_hours:
                    thread_id += 1
            records.append({
                'hotel_id': hotel_id,
                'booking_id': booking_id,
                'thread_id': thread_id,
                'message': row['message'],
                'date_add': row['date_add'],
            })
            prev_time = row['date_add']
    return pd.DataFrame(records)

df_msgs = detect_threads(df)
print(f"Сообщений с thread_id: {len(df_msgs)}")

# Сворачиваем в треды — один ряд = один тред
threads = (
    df_msgs.groupby(['hotel_id', 'booking_id', 'thread_id'])
    .agg(
        text=('message', lambda msgs: '\n---\n'.join(msgs)),
        n_guest_msgs=('message', 'count'),
        thread_start=('date_add', 'min'),
        thread_end=('date_add', 'max'),
    )
    .reset_index()
)

print(f"Всего тредов: {len(threads)}")
print(threads.groupby('hotel_id')['thread_id'].count().rename(index=HOTEL_NAMES).rename('threads'))

Сообщений с thread_id: 24021
Всего тредов: 12279
hotel_id
БС74      7003
МК16      2758
You&Co    2468
М73         50
Name: threads, dtype: int64


## 3. GPT-4o-mini Классификация

Каждый тред → PROBLEM / QUESTION / OTHER + reason + confidence.

**Le Wagon:** Data Toolkit → Using AI · batching 20 тредов за один вызов (~20x дешевле)

In [5]:
import os
from openai import OpenAI

def classify_batch(client, batch_texts):
    """
    Классифицируем список тредов одним GPT-запросом.
    Возвращает список: [{category, reason, confidence}, ...]
    """
    numbered = '\n\n'.join(
        f'[{i+1}]\n{text[:800]}' for i, text in enumerate(batch_texts)
    )

    prompt = f"""Ты — аналитик отеля. Классифицируй каждый тред гостевых сообщений.

КАТЕГОРИИ:
- PROBLEM: гость сообщает о проблеме, которую нужно физически устранить
- QUESTION: вопрос, запрос информации или услуги
- OTHER: благодарность, технический мусор, непонятный текст

Для каждого треда ответь строго в формате (одна строка):
N|CATEGORY|краткая причина на русском (до 10 слов)|confidence 0.0-1.0

{numbered}"""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{{"role": "user", "content": prompt}}],
        temperature=0,
        max_tokens=1500,
    )

    results = []
    for line in resp.choices[0].message.content.strip().split('\n'):
        parts = line.split('|')
        if len(parts) >= 4:
            try:
                results.append({{
                    'category': parts[1].strip(),
                    'reason': parts[2].strip(),
                    'confidence': float(parts[3].strip()),
                }})
            except (ValueError, IndexError):
                results.append({{'category': 'OTHER', 'reason': 'parse error', 'confidence': 0.5}})
        elif len(parts) == 3:
            results.append({{'category': parts[1].strip(), 'reason': parts[2].strip(), 'confidence': 0.8}})

    # Если GPT вернул неверное число строк — заполняем OTHER
    while len(results) < len(batch_texts):
        results.append({{'category': 'OTHER', 'reason': 'missing', 'confidence': 0.5}})

    return results[:len(batch_texts)]

In [6]:
# ── Запуск классификации ──────────────────────────────────────────────────────
# Если уже есть threads_classified.parquet — пропускаем этот шаг
CLASSIFIED_PATH = "data/threads_classified_full.parquet"

if Path(CLASSIFIED_PATH).exists():
    threads = pd.read_parquet(CLASSIFIED_PATH)
    print(f"Загружен готовый файл: {CLASSIFIED_PATH}")
    print(threads['category'].value_counts())
else:
    import os
    key = OPENAI_API_KEY or os.environ.get("OPENAI_API_KEY")
    assert key, "Нужен OPENAI_API_KEY"
    client = OpenAI(api_key=key)

    texts = threads['text'].tolist()
    categories, reasons, confidences = [], [], []

    print(f"Классифицируем {len(texts)} тредов батчами по {BATCH_SIZE}...")
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        results = classify_batch(client, batch)
        for r in results:
            categories.append(r['category'])
            reasons.append(r['reason'])
            confidences.append(r['confidence'])
        if (i // BATCH_SIZE) % 10 == 0:
            done = min(i + BATCH_SIZE, len(texts))
            print(f"  {done}/{len(texts)} тредов...")

    threads['category']   = categories
    threads['reason']     = reasons
    threads['confidence'] = confidences

    threads.to_parquet(CLASSIFIED_PATH, index=False)
    print(f"\nСохранено: {CLASSIFIED_PATH}")
    print(threads['category'].value_counts())

Классифицируем 12279 тредов батчами по 20...


TypeError: unhashable type: 'dict'

## 4. Сводка по отелям

In [ ]:
# ── Breakdown по отелям ───────────────────────────────────────────────────────
breakdown = (
    threads.groupby(['hotel_id', 'category'])
    .size()
    .unstack(fill_value=0)
    .rename(index=HOTEL_NAMES)
)
# Процент
breakdown_pct = (breakdown.T / breakdown.sum(axis=1)).T * 100

print("Треды по отелям и категориям:")
print(breakdown)
print("\n%:")
print(breakdown_pct.round(1))

## 5. TF-IDF + KMeans: поиск лучшего k

**Le Wagon:** Machine Learning → TF-IDF (2.3) + KMeans (2.2) + Silhouette (2.3)

Теперь на полном датасете (все отели) — больше данных = стабильнее кластеры.

In [ ]:
# ── Стоп-слова ────────────────────────────────────────────────────────────────
STOP_WORDS = set("""
и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по
только ее мне было вот от меня еще нет о ней ему теперь когда даже ну вдруг ли
если уже или ни быть был нибудь раз уж вам вас их бы них она они оно мы ими им
будет будь будьте этот того без может мог бы лишь один себя себе чем чтоб чтобы
ведь другой кто тут там потом наш при где более всех этих после этом два какой
мой свой свою тебя тебе тогда та то ту те до ещё мои мою мое моя моей моего
нас для нее над нам об так из за через очень все сами весь вся всю всего всех
этому этой всей этих того этот тоже каждый каждая каждое какой который
здравствуйте здравствуй привет добрый день вечер утро доброе доброго
пожалуйста спасибо ок хорошо ладно понял понятно сделать
""".split())

morph = pymorphy3.MorphAnalyzer()

def preprocess(text: str) -> str:
    text = text.replace('---', ' ').lower()
    text = re.sub(r'[^а-яё\s]', ' ', text)   # убираем латиницу, цифры, HTML-остатки
    tokens = text.split()
    lemmas = []
    for t in tokens:
        if len(t) < 3:
            continue
        lemma = morph.parse(t)[0].normal_form
        if lemma not in STOP_WORDS and len(lemma) >= 3:
            lemmas.append(lemma)
    return ' '.join(lemmas)

def run_clustering(texts_clean, k):
    vec = TfidfVectorizer(max_features=500, ngram_range=(1,2), min_df=2, sublinear_tf=True)
    X = vec.fit_transform(texts_clean)
    terms = vec.get_feature_names_out()

    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels, metric='cosine', sample_size=min(500, len(texts_clean)))

    topics = []
    for i, center in enumerate(km.cluster_centers_):
        top_idx = center.argsort()[-5:][::-1]
        top_terms = [terms[j] for j in top_idx]
        idx = np.where(labels == i)[0]
        topics.append({'id': i+1, 'size': int(len(idx)),
                       'top_terms': list(top_terms),
                       'samples': [texts_clean[j][:120] for j in idx[:2]]})

    max_pct = max(t['size'] for t in topics) / len(texts_clean)
    tiny    = sum(1 for t in topics if t['size'] < len(texts_clean) * 0.02)
    return {'k': k, 'silhouette': sil, 'inertia': km.inertia_,
            'max_cluster_pct': max_pct, 'tiny_clusters': tiny, 'topics': topics}

In [ ]:
# ── Препроцессинг + перебор k ─────────────────────────────────────────────────
all_results = {}

for cat in CATEGORIES:
    print(f"\n{'═'*60}")
    print(f"  {cat}")
    print(f"{'═'*60}")

    texts = threads[threads['category'] == cat]['text'].tolist()
    texts_clean = [preprocess(t) for t in texts]
    texts_clean = [t for t in texts_clean if len(t.strip()) > 0]
    print(f"  Тредов: {len(texts_clean)}")

    results = []
    for k in K_RANGE:
        r = run_clustering(texts_clean, k)
        results.append(r)
        flag = ''
        if r['max_cluster_pct'] > 0.35: flag += ' ⚠️ catchall'
        if r['tiny_clusters'] > 0:      flag += f' ⚠️ {r["tiny_clusters"]} tiny'
        print(f"  k={k:2d}  sil={r['silhouette']:.3f}  "
              f"max={r['max_cluster_pct']:.0%}  tiny={r['tiny_clusters']}{flag}")

    all_results[cat] = {'texts_clean': texts_clean, 'results': results}

## 6. Silhouette + Elbow графики

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Выбор k: Silhouette + Elbow', fontsize=14)

for idx, cat in enumerate(CATEGORIES):
    results = all_results[cat]['results']
    ks      = [r['k'] for r in results]
    sils    = [r['silhouette'] for r in results]
    inertias= [r['inertia'] for r in results]
    max_pcts= [r['max_cluster_pct'] for r in results]

    adj = [s - 0.05*(p > 0.35) - 0.02*r['tiny_clusters']
           for s, p, r in zip(sils, max_pcts, results)]
    best_k = ks[int(np.argmax(adj))]

    # Silhouette
    ax = axes[idx, 0]
    ax.plot(ks, sils, 'o-', color='steelblue')
    for k, s, p in zip(ks, sils, max_pcts):
        if p > 0.35:
            ax.plot(k, s, 's', color='orange', markersize=10, zorder=5)
    ax.axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'best k={best_k}')
    ax.set_title(f'{cat} — Silhouette (🟧=catchall)')
    ax.set_xlabel('k'); ax.set_ylabel('Silhouette')
    ax.legend(); ax.grid(True, alpha=0.3)

    # Elbow
    ax = axes[idx, 1]
    ax.plot(ks, inertias, 'o-', color='coral')
    ax.axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'best k={best_k}')
    ax.set_title(f'{cat} — Inertia (Elbow)')
    ax.set_xlabel('k'); ax.set_ylabel('Inertia')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. GPT оценивает когерентность тем

Все k за один запрос → GPT смотрит на дубли, catchall, мусор → выбирает лучший k.

In [ ]:
def evaluate_with_gpt(results_list, category, api_key=None):
    import os
    from openai import OpenAI
    key = api_key or OPENAI_API_KEY or os.environ.get("OPENAI_API_KEY")
    if not key:
        print("  ⚠️  Нет OPENAI_API_KEY"); return None
    client = OpenAI(api_key=key)

    lines = []
    for r in results_list:
        lines.append(f"\n--- k={r['k']} (sil={r['silhouette']:.3f}) ---")
        for t in r['topics']:
            lines.append(f"  Тема {t['id']} ({t['size']} тредов): {', '.join(t['top_terms'])}")

    prompt = f"""Ты — аналитик отеля. Оцени варианты кластеризации гостевых сообщений
(категория: {category}). Слова лемматизированы.

{''.join(lines)}

Для каждого k (score 1-10):
- 10 = чёткие операционные темы, нет дублей, нет мусора
- Штраф за: дубли одной темы, catchall >30%, мусорные кластеры

Ответь ТОЛЬКО JSON:
{{
  "evaluations": [{{"k": 5, "score": 6, "issues": "..."}},...],
  "best_k": 11,
  "best_k_reason": "..."
}}"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{{"role": "user", "content": prompt}}],
            temperature=0.1, max_tokens=2000,
        )
        text = resp.choices[0].message.content
        text = re.sub(r'```json\s*|```\s*', '', text).strip()
        return json.loads(text)
    except Exception as e:
        print(f"  ⚠️  GPT ошибка: {e}"); return None

In [ ]:
# ── Запуск GPT оценки ─────────────────────────────────────────────────────────
gpt_evals = {}

for cat in CATEGORIES:
    print(f"\n🤖 GPT оценка: {cat}")
    gpt = evaluate_with_gpt(all_results[cat]['results'], cat)
    gpt_evals[cat] = gpt
    if gpt:
        print(f"  best_k = {gpt['best_k']}")
        print(f"  {gpt['best_k_reason']}")
        for ev in gpt['evaluations']:
            bar = '█' * ev['score'] + '░' * (10 - ev['score'])
            print(f"  k={ev['k']:2d} [{bar}] {ev['score']}/10  {ev.get('issues','')}")

## 8. Финальные темы + визуализация

In [ ]:
# ── Финальный выбор ────────────────────────────────────────────────────────────
final = {}

for cat in CATEGORIES:
    results = all_results[cat]['results']
    gpt     = gpt_evals.get(cat)

    if gpt and gpt.get('best_k'):
        best_k = gpt['best_k']; method = 'GPT'
    else:
        adj = [r['silhouette'] - 0.05*(r['max_cluster_pct'] > 0.35)
               - 0.02*r['tiny_clusters'] for r in results]
        best_k = results[int(np.argmax(adj))]['k']; method = 'Silhouette'

    best_r = next(r for r in results if r['k'] == best_k)

    print(f"\n{'═'*60}")
    print(f"  ✅ {cat}: k={best_k} ({method}), sil={best_r['silhouette']:.3f}")
    print(f"{'═'*60}")
    for t in sorted(best_r['topics'], key=lambda x: -x['size']):
        bar = '█' * max(1, t['size'] // max(1, len(all_results[cat]['texts_clean']) // 40))
        print(f"  {t['id']:2d}. ({t['size']:4d}) {bar}")
        print(f"       {' | '.join(t['top_terms'])}")

    final[cat] = {'best_k': best_k, 'method': method,
                  'silhouette': best_r['silhouette'],
                  'topics': best_r['topics']}

In [ ]:
# ── Барчарт финальных тем ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Итоговые темы (все отели)', fontsize=14)

for idx, cat in enumerate(CATEGORIES):
    ax = axes[idx]
    topics = sorted(final[cat]['topics'], key=lambda t: t['size'], reverse=True)
    labels = [' | '.join(t['top_terms'][:3]) for t in topics]
    sizes  = [t['size'] for t in topics]

    bars = ax.barh(range(len(labels)), sizes, color='steelblue', alpha=0.8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Кол-во тредов')
    ax.set_title(f"{cat} — k={final[cat]['best_k']} ({final[cat]['method']})")
    for bar, s in zip(bars, sizes):
        ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
                str(s), va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 9. Сохранение

In [ ]:
# ── Сохраняем результаты ──────────────────────────────────────────────────────
out = {}
for cat in CATEGORIES:
    out[cat] = {
        'best_k': final[cat]['best_k'],
        'method': final[cat]['method'],
        'silhouette': round(final[cat]['silhouette'], 4),
        'topics': [{'id': t['id'], 'size': int(t['size']),
                    'top_terms': list(t['top_terms'])} for t in final[cat]['topics']],
    }

Path("data/topic_results_full.json").write_text(
    json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8'
)
print("💾 data/topic_results_full.json")
print(json.dumps(out, ensure_ascii=False, indent=2))